[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/07_hnet_tokenizer_transfer/07_hnet_tokenizer_transfer.ipynb)

# Módulo 07 · Tokenizer-free II (H-Net) y *tokenizer transfer*

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 6 h (lección ≈ 3.5 h + proyecto ≈ 2.5 h) |
| **GPU recomendada** | `FAST_DEV_RUN=True`: **CPU** (≈ 12-15 min en total). Modo completo: **L4** recomendada (≈ 6-10 unidades de Colab, estimación) para ajustar un GPT-2 pequeño real; T4 sirve con lotes menores |
| **Prerrequisitos** | 02-04 (BPE, vocabularios), 05 (patologías), **06 (BLT y parches por entropía)**; adelanto del 09 (matriz de embeddings y weight tying) |
| **Madurez del tema** | H-Net 🔴 (arXiv 2025, ICLR 2026, ≤1.3B de FLOPs equivalentes) · tokenizer transfer 🟡 (varias técnicas publicadas y código abierto; sin receta única) |

> **Contexto Políglota Labs.** Dos preguntas llegan a tu mesa. (1) *Producto:* «¿merece la pena apostar por un modelo sin tokenizer, en el que el **segmentador se aprende**?» (2) *Costes:* «Tenemos un modelo
> pequeño preentrenado con un tokenizer en inglés y código; el español sale en el doble de tokens. ¿Podemos **cambiarle el tokenizer** sin reentrenar de cero?» Esta lección responde a ambas con código que corre en CPU
> y con una dosis de escepticismo: lo que reproduces es **didáctico y diminuto**; las cifras de los papers son de los autores.

## 1. Objetivos de aprendizaje

1. **Explicar** el *dynamic chunking* de H-Net: módulo de *routing*, *downsampler*, suavizado (*smoothing*), *upsampler* con *straight-through estimator* y **ratio loss**, y por qué cada pieza hace falta.
2. **Implementar** un H-Net de una etapa en miniatura (PyTorch) y **verificar** su causalidad; **observar** cómo el modelo aprende fronteras sin supervisión.
3. **Comparar** el chunking aprendido con la heurística de espacios (SpaceByte) y con *pooling* fijo, y con el patching por entropía del módulo 06.
4. **Argumentar** con evidencia a favor y en contra de «¿está muerta la tokenización?», distinguiendo lo que los papers prueban de lo que no.
5. **Clasificar** las técnicas de *tokenizer transfer* (FVT, WECHSEL, FOCUS, Trans-Tokenization, ZeTT) por lo que necesitan y lo que hacen.
6. **Implementar** inicializaciones de embeddings para vocabulario nuevo (aleatoria, media, Hewitt, FVT, FOCUS-lite, WECHSEL-lite) y **medir** su efecto en bits-por-byte antes y después de un ajuste corto.
7. **Diseñar** un protocolo de evaluación honesto para un cambio de tokenizer (bpb, no pérdida por token).

## 2. 🔁 Conexión con módulos anteriores

<details>
<summary>1 · (Módulo 06) ¿Qué decide dónde se corta en BLT y qué lo hace no diferenciable?</summary>

Un modelo de bytes **aparte** (el modelo de entropía) mide $H(x_i)$ y un umbral $\theta$ decide: es una decisión discreta tomada *antes* y fuera del grafo de la red principal, por lo que el segmentador no recibe gradiente del objetivo de lenguaje. H-Net quiere que esa decisión se aprenda de extremo a extremo.

</details>

<details>
<summary>2 · (Módulo 02/04) ¿Por qué no puedes comparar la pérdida por token de dos tokenizers distintos?</summary>

Porque cada token cubre un número distinto de bytes: un tokenizer con tokens más largos tiene mayor pérdida *por token* aunque modele mejor el texto. Hay que normalizar por bytes (bpb) — idea que usamos otra vez aquí.

</details>

<details>
<summary>3 · (Módulo 05) Un token del vocabulario casi no aparece en el corpus de entrenamiento del modelo. ¿Qué le pasa a su embedding y qué implicaría para tokens recién añadidos?</summary>

Sus filas de embedding reciben pocos gradientes y quedan cerca de la inicialización (tokens sub-entrenados/glitch). Un token **nuevo** tiene el mismo riesgo: si su inicialización es mala, el modelo tarda en aprenderlo o lo trata como ruido; por eso la inicialización importa.

</details>

## 3. Intuición primero

**Del módulo 06 al 07.** BLT demostró que agrupar bytes en parches funciona, pero con una pieza *externa y heurística*: un modelo de entropía entrenado aparte y un umbral. Pregunta natural: *¿y si el modelo aprendiera por sí solo dónde cortar, y además a varios niveles (bytes → palabras → frases)?* Eso es **H-Net** (Hwang, Wang y Gu, 2025): una red jerárquica tipo **U-Net** cuyo *downsampling* es **dinámico y aprendido** junto con el resto del modelo.

* **Analogía de compresión.** Una cadena de compresores donde cada etapa decide qué muestras conservar. Lo difícil es entrenar una decisión *discreta* («conservo este vector o no») con descenso de gradiente. H-Net lo consigue con tres trucos: un *router* basado en cambio de contexto, un **suavizado** que hace diferenciable la reconstrucción y una **ratio loss** que evita el colapso (todo o nada).
* **Analogía de recsys.** Es como un *mixture-of-experts* o *mixture-of-depths*: un router decide qué elementos reciben cómputo caro; la pérdida de balanceo impide que el router se degenere. De hecho la ratio loss se inspira en la de balanceo de carga de MoE.

**Segunda mitad de la lección — «tokenizer transfer».** Aunque el futuro fuera sin tokenizer, hoy hay millones de parámetros ya entrenados *atados* a un vocabulario. Cambiarlo (idioma, dominio, eficiencia) exige **inicializar los embeddings del vocabulario nuevo**. Es el primo pragmático del tema: menos glamuroso, mucho más usado.

> ### 🎨 GEMINI-IMAGEN · ID: `M07-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **H-Net: U-Net jerárquica con chunking dinámico**
>
> **Qué muestra:** Arquitectura de H-Net de dos etapas: encoder → chunking → (encoder → chunking → main) → dechunk → decoder, con las conexiones residuales.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Diagrama en U de H-Net con dos etapas de chunking, red principal en el fondo y conexiones residuales
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio de arquitectura, fondo blanco, estilo figura de paper (sans-serif legible, flechas claras, sin ilustraciones decorativas ni texto inventado). Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Formato 16:9.
> Título: "H-Net: una U-Net con chunking dinámico (2 etapas)".
> Composición en forma de "U" con las entradas a la izquierda y las salidas a la derecha.
> Bajada (izquierda, de arriba abajo), cada bloque con su etiqueta: "Bytes de entrada (L₀ posiciones)" → bloque azul "Encoder E⁰" → bloque naranja pequeño "Chunking 0 (routing + downsampler)" → "Vectores de etapa 1 (L₁ < L₀)" → bloque azul "Encoder E¹" → bloque naranja pequeño "Chunking 1" → "Vectores de etapa 2 (L₂ < L₁)".
> Fondo de la U (abajo): bloque verde grande "Main network M (Transformer)" con la nota "procesa pocos vectores → casi todos los parámetros".
> Subida (derecha, de abajo arriba): bloque naranja pequeño "Dechunking 1 (smoothing + upsampler)" → bloque azul "Decoder D¹" → bloque naranja pequeño "Dechunking 0" → bloque azul "Decoder D⁰" → "Predicción del siguiente byte".
> Dos flechas horizontales discontinuas grises etiquetadas "conexión residual (Linear)" que van del Encoder E⁰ al Decoder D⁰ y del Encoder E¹ al Decoder D¹.
> Una anotación lateral: "E y D: Mamba-2 en el paper · M: Transformer". Otra: "las fronteras (chunks) se aprenden de extremo a extremo".
> Evita: ilustraciones decorativas, texto ilegible, más etapas de las indicadas, logotipos, flechas ambiguas, números no indicados.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M07-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Diagrama en U de H-Net con dos etapas de chunking, red principal en el fondo y conexiones residuales](assets/M07-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> bytes -> [E0] --chunk0--> [E1] --chunk1--> [ M (Transformer) ] --dechunk1--> [D1] --dechunk0--> [D0] -> siguiente byte
>             \___________residual (Linear)____________________________________________/ (E0 -> D0)
>                          \_______residual (Linear)____________/ (E1 -> D1)
> ```

> **👀 Qué debes observar**
>
> Las etapas **externas** (E, D) trabajan sobre secuencias largas, así que deben ser baratas (Mamba-2 en el paper); la **red principal** M trabaja sobre pocos vectores y concentra los parámetros. Las conexiones residuales devuelven al decoder el detalle fino que se perdió al comprimir. Una H-Net de una etapa es la misma figura con un solo U.

In [ ]:
!pip install -q torch numpy matplotlib tokenizers transformers datasets scipy

In [ ]:
import os, sys, time, math, random, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
FAST_DEV_RUN = True      # True: CPU, minutos. False: GPU (T4/L4), escala completa
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", min(4, os.cpu_count() or 1))))
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN, "| torch", torch.__version__)

import re
import matplotlib.pyplot as plt
import torch.nn as nn, torch.nn.functional as F
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, VERDE, GRIS = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280"

import copy, re
from collections import Counter

### 3.1 Corpus Políglota (utilidades del módulo 00)

Implementación **canónica** de `carga_corpus()` (autor A1): Wikipedia ES/EN (HF) → API de Wikipedia → texto **sintético**; código: CodeSearchNet → funciones **reales** de la stdlib. Si ves «SINTÉTICO», interpreta las tendencias, no las cifras absolutas.

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
corp = carga_corpus(FAST_DEV_RUN)
print("fuente del corpus:", corp["source"], "| docs:", len(corp["docs"]))
rng = random.Random(SEED)
docs = corp["docs"][:]; rng.shuffle(docs)
cut = int(0.9 * len(docs))
DOCS_TR, DOCS_VA = docs[:cut], docs[cut:]
MAX_CHARS = 1500 if FAST_DEV_RUN else 6000
LIMITE_BYTES = None if FAST_DEV_RUN else 3_000_000        # el modelo n-grama es Python puro: acota el modo completo

def texto(ds):
    return "\n\n".join(d["text"][:MAX_CHARS] for d in ds)

TXT_TR, TXT_VA = texto(DOCS_TR), texto(DOCS_VA)
TR = np.frombuffer(TXT_TR.encode("utf-8"), dtype=np.uint8).copy()[:LIMITE_BYTES]
VA = np.frombuffer(TXT_VA.encode("utf-8"), dtype=np.uint8).copy()[:LIMITE_BYTES]
print(f"bytes train={len(TR):,} | val={len(VA):,}")
for lang in ["es", "en", "code"]:
    t = "".join(d["text"][:MAX_CHARS] for d in DOCS_TR if d["lang"] == lang)
    print(f"{lang:>4}: {len(t):>8,} chars -> {len(t.encode()):>8,} bytes  ({len(t.encode())/max(1, len(t)):.3f} bytes/char)")

## 4. Teoría: el *dynamic chunking* de H-Net

Una etapa de H-Net aplica $\hat x = \mathcal{E}(x)$, $(x', p) = \text{Chunk}(\hat x)$, $\hat z' = \mathcal{M}(x')$, $z = \text{Dechunk}(\hat z', p) + \text{Linear}(\hat x)$ y decodifica con $\mathcal{D}$. Los símbolos: $x\in\mathbb{R}^{L\times D}$ la entrada de la etapa, $\hat x$ la salida del encoder, $p_t\in[0,1]$ la «confianza de frontera» en $t$.

**(1) Routing.** Medir *cambio de contexto* entre vecinos con similitud coseno de proyecciones aprendidas:

$$q_t = W_q\hat x_t,\quad k_t = W_k\hat x_t,\quad p_t = \tfrac12\!\left(1 - \frac{q_t^\top k_{t-1}}{\lVert q_t\rVert\,\lVert k_{t-1}\rVert}\right),\quad b_t = \mathbb{1}[p_t \ge 0.5],\quad p_1 = 1.$$

Si dos posiciones consecutivas son parecidas ($\cos\approx 1$) $p_t\approx 0$ (misma unidad); si difieren ($\cos\approx -1$) $p_t\approx 1$ (frontera). $b_t$ marca los bordes.

**(2) Downsampler.** Se **seleccionan** los vectores con $b_t=1$ (el paper compara con *mean/max pooling* y atención cruzada y se queda con la selección por simplicidad y rendimiento). La red principal $\mathcal{M}$ ve solo esos vectores.

**(3) Smoothing (dechunk).** La decisión $b_t$ es discreta y bloquea el gradiente. El suavizado hace la reconstrucción **diferenciable** con una media móvil exponencial sobre los *chunks*:

$$\bar z_j = P_j\,\hat z_j + (1-P_j)\,\bar z_{j-1},$$

con $P_j$ la $p$ del vector-frontera del chunk $j$. Con $P_j\approx 1$ el chunk se queda tal cual; con $P_j\approx 0.5$ se mezcla con el anterior (corrige fronteras dudosas).

**(4) Upsampler con STE.** Cada $\bar z_j$ se repite hasta la siguiente frontera ($\tilde z_t=\bar z_{\sum_{k\le t}b_k}$) y se multiplica por la confianza:

$$c_t = p_t^{\,b_t}(1-p_t)^{1-b_t},\qquad \text{STE}(c_t)=c_t+\operatorname{sg}(1-c_t),\qquad \text{Upsample}_t = \text{STE}(c_t)\,\tilde z_t,$$

donde $\operatorname{sg}$ es *stop-gradient*: en forward vale 1 (no altera la señal) y en backward deja pasar el gradiente de $c_t$. Premia fronteras confiadas y acertadas, y castiga las dudosas.

**(5) Ratio loss.** Sin regularizar, el router colapsaría a «todo es frontera» o «nada lo es». Con $F=\frac1L\sum b_t$ (fracción real de bordes) y $G=\frac1L\sum p_t$ (probabilidad media):

$$\mathcal{L}_{\text{ratio}} = \frac{N}{N-1}\Big((N-1)\,F\,G + (1-F)(1-G)\Big),\qquad \mathcal{L}=\mathcal{L}_{AR}+\alpha\sum_s \mathcal{L}^{s}_{\text{ratio}},\ \ \alpha=0.03,$$

con $N$ el factor de compresión objetivo. Aunque $F$ no es diferenciable, el gradiente fluye por $G$; el mínimo $\mathcal{L}_{\text{ratio}}=1$ se alcanza en $F=G=1/N$. (Idea tomada de la pérdida de balanceo de carga de los MoE.)

**Ejemplo a mano.** *Routing:* si $\cos(q_t,k_{t-1})=0.8$, $p_t=\tfrac12(1-0.8)=0.10$ → $b_t=0$ (sigue la unidad); si $\cos=-0.2$, $p_t=0.60$ → $b_t=1$ (frontera). *Ratio loss* con $N=4$ (objetivo: 1 frontera cada 4 bytes): en el óptimo $F=G=0.25$, $\mathcal{L}_{\text{ratio}}=\tfrac43(3\cdot0.0625+0.75\cdot0.75)=\tfrac43\cdot0.75=\mathbf{1}$; si el router colapsa a «todo es frontera» ($F=G=1$), $\tfrac43\cdot3=\mathbf{4}$; a «nada es frontera» ($F=G=0$), $\tfrac43\cdot1=\mathbf{1.33}$. Ambos colapsos se penalizan, como la pérdida de balanceo de un MoE penaliza mandar todo a un experto.

**Mejoras de entrenamiento** (§2.3 del paper): normalización al final de cada red (*norm balance*), proyección lineal solo en el camino residual, y **tasas de aprendizaje mayores en las etapas externas** (que ven secuencias más largas y deciden las fronteras de las internas).

**Inferencia.** H-Net decide *por byte* si lo pasa a la red principal; el decoder corre en cada byte y la red principal solo al cerrar chunks — parecido a la decodificación especulativa (un modelo pequeño cada paso, uno grande cada pocos).

> ### 🎨 GEMINI-IMAGEN · ID: `M07-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Routing, smoothing y upsampling en H-Net**
>
> **Qué muestra:** Mecánica de una etapa de chunking: similitud coseno entre vecinos → p_t → bordes → selección → suavizado EMA → repetición hasta el siguiente borde.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Cuatro paneles que muestran routing por similitud coseno, selección de bordes, suavizado EMA y repetición con STE
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, estilo figura de paper (sans-serif legible, flechas claras, sin texto inventado). Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Formato 16:9.
> Título: "Una etapa de dynamic chunking, paso a paso".
> Cuatro paneles en fila de izquierda a derecha, numerados 1 a 4, todos sobre la secuencia de bytes "n e w _ p r o d u c t !" (12 casillas; "_" es un espacio).
> Panel 1 "Routing": debajo de cada par de casillas vecinas, una barra azul cuya altura es p_t = ½(1 − cos(q_t, k_{t−1})). Barras altas (≥ 0.5, resaltadas en naranja) bajo "n", "p" y bajo "d" (esta última algo más baja, ≈ 0.55). Línea horizontal discontinua roja en 0.5 etiquetada "umbral 0.5". La primera barra vale 1.0 (etiqueta "p₁ = 1").
> Panel 2 "Downsampler": solo las casillas con barra naranja (n, p, d) se copian a una fila nueva más corta, etiquetada "vectores seleccionados x′ (L′ < L)".
> Panel 3 "Smoothing": tres círculos de salida de la red principal con la fórmula "z̄ⱼ = Pⱼ·ẑⱼ + (1 − Pⱼ)·z̄ⱼ₋₁"; el tercer círculo, con P = 0.55, aparece mezclado (degradado entre verde y gris) y se anota "frontera dudosa → se mezcla con el chunk anterior".
> Panel 4 "Upsampler": cada círculo se repite hasta el siguiente borde sobre las 12 casillas (3 bloques de colores: azul, naranja, verde de anchos 4, 5 y 3), con la leyenda "× STE(cₜ)" y "forward vale 1; el gradiente pasa por cₜ".
> Evita: ilustraciones decorativas, texto ilegible, números no indicados, flechas ambiguas, sombras 3D.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M07-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Cuatro paneles que muestran routing por similitud coseno, selección de bordes, suavizado EMA y repetición con STE](assets/M07-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> 1 Routing    : n e w _ p r o d u c t !   p_t: [1.0 .1 .1 .2 .9 .1 .1 .55 .1 .1 .1 .1]  borde si p>=0.5
> 2 Downsample : se conservan los vectores en n, p, d   -> L' = 3
> 3 Smoothing  : z̄_j = P_j ẑ_j + (1-P_j) z̄_{j-1}      (d con P=.55 se mezcla con el chunk anterior)
> 4 Upsample   : [n e w _][p r o d u][c t !] x STE(c_t)   (forward: x1; backward: gradiente de c_t)
> ```

> **👀 Qué debes observar**
>
> El suavizado es lo que permite aprender fronteras «a ciegas»: una frontera dudosa (P≈0.5) no tira la información del chunk, la **mezcla** con el anterior; y, si acierta, el factor $c_t$ la recompensa. Sin esto, el gradiente no llegaría a $W_q,W_k$.

### 4.1 BLT frente a H-Net (qué cambia respecto al módulo 06)

| | **BLT (módulo 06)** | **H-Net (este módulo)** |
|---|---|---|
| ¿Quién decide las fronteras? | Modelo de entropía **aparte**, umbral fijo | Router **dentro** del modelo, entrenado extremo a extremo |
| ¿Diferenciable? | No (preprocesado) | Sí (smoothing + STE + ratio loss) |
| Control del coste | Umbral $\theta$ (también en inferencia) | Compresión objetivo $N$ (ratio loss) |
| Representación del chunk | Atención cruzada sobre los bytes del parche | **Selección** del vector-frontera (+ residual al decoder) |
| Jerarquía | 1 nivel de parches | **Recursiva**: la red principal puede ser otra H-Net |
| Backbone | Transformers | Mamba-2 en E/D, Transformer en M |
| Escala reportada | hasta 8B / 4T bytes | FLOPs de GPT-3 Large y XL (760M / 1.3B), 100B tokens de FineWeb-Edu |

## 5. MiniHNet desde cero

**Qué es y qué no es.** Una etapa de H-Net con E y D como transformers causales pequeños (el paper usa Mamba-2) y los mismos mecanismos de chunking: routing por coseno, selección, EMA, upsampler con STE, residual proyectado inicializado a 0 y ratio loss. No incluye jerarquía de dos etapas, *norm balance* completo ni *LR modulation*. Comprobamos la **causalidad** igual que en el módulo 06.

In [ ]:
def mascara_causal(T, dev):
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=dev), 1)

def capas(d, L, H=4):
    layer = nn.TransformerEncoderLayer(d, H, 4 * d, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
    return nn.TransformerEncoder(layer, L, enable_nested_tensor=False)

ES_ESPACIO = torch.tensor([not (48 <= b <= 57 or 65 <= b <= 90 or 97 <= b <= 122 or b >= 0x80) for b in range(256)])

def inicios_espacio_t(x):
    """[B,T] bool: el byte abre palabra (el anterior es 'espacio' y este no). Es la heurística de SpaceByte."""
    prev = F.pad(x, (1, 0), value=32)[:, :-1]
    return ES_ESPACIO[prev] & ~ES_ESPACIO[x]

def lote(data, B, T, g):
    ix = torch.randint(0, len(data) - T - 1, (B,), generator=g)
    x = torch.stack([torch.from_numpy(data[i:i + T].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + T + 1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

In [ ]:
class MiniHNet(nn.Module):
    """H-Net de UNA etapa, en miniatura. Sigue el diseño de Hwang, Wang y Gu (2025):
       E (causal) -> routing (coseno entre vecinos) -> selección de bordes -> M (sobre chunks) -> dechunk (suavizado EMA + STE) -> D.
       Simplificación: E y D son transformers causales (el paper usa Mamba-2); modo 'space'/'pool' = chunking NO aprendido."""
    def __init__(self, d=64, d_main=128, Le=1, Lm=3, Ld=2, ctx=128, modo="dc"):
        super().__init__()
        self.d, self.modo = d, modo
        self.emb, self.pos = nn.Embedding(256, d), nn.Embedding(ctx, d)
        self.enc, self.ne = capas(d, Le), nn.LayerNorm(d)
        self.wq, self.wk = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.pad = nn.Parameter(torch.zeros(d_main - d))              # ensancha d -> d_main con un vector compartido (como SpaceByte/H-Net)
        self.main, self.nm = capas(d_main, Lm), nn.LayerNorm(d_main)
        self.res = nn.Linear(d, d); nn.init.zeros_(self.res.weight); nn.init.zeros_(self.res.bias)   # residual proyectado, init ~0
        self.dec, self.ln, self.head = capas(d, Ld), nn.LayerNorm(d), nn.Linear(d, 256, bias=False)

    def fronteras(self, xh, x):
        B, T, _ = xh.shape
        if self.modo == "dc":                                          # routing module: p_t = (1 - cos(q_t, k_{t-1})) / 2
            cos = F.cosine_similarity(self.wq(xh[:, 1:]), self.wk(xh[:, :-1]), dim=-1)
            p = torch.cat([torch.ones(B, 1, device=xh.device), 0.5 * (1 - cos)], 1).clamp(0, 1)
            return p, p >= 0.5
        b = inicios_espacio_t(x) if self.modo == "space" else (torch.arange(T, device=xh.device) % 4 == 0).expand(B, T)
        b = b.clone(); b[:, 0] = True
        return b.float(), b

    def forward(self, x):
        B, T = x.shape; dev = x.device
        xh = self.ne(self.enc(self.emb(x) + self.pos(torch.arange(T, device=dev)), mask=mascara_causal(T, dev), is_causal=True))
        p, b = self.fronteras(xh, x)
        rank = b.long().cumsum(1) - 1; Lc = int(rank.max()) + 1       # índice de chunk de cada byte
        bi, ti = b.nonzero(as_tuple=True)
        sel = xh.new_zeros(B, Lc, self.d); sel[bi, rank[bi, ti]] = xh[bi, ti]      # downsampler: se queda con los vectores-frontera
        P = xh.new_zeros(B, Lc); P[bi, rank[bi, ti]] = p[bi, ti]
        z = self.nm(self.main(torch.cat([sel, self.pad.expand(B, Lc, -1)], -1), mask=mascara_causal(Lc, dev), is_causal=True))[..., :self.d]
        prev, zb = xh.new_zeros(B, self.d), []
        for j in range(Lc):                                            # suavizado EMA:  z̄_j = P_j z_j + (1 - P_j) z̄_{j-1}
            prev = P[:, j:j + 1] * z[:, j] + (1 - P[:, j:j + 1]) * prev; zb.append(prev)
        zt = torch.stack(zb, 1).gather(1, rank.unsqueeze(-1).expand(-1, -1, self.d))    # upsampler: repite hasta la siguiente frontera
        c = torch.where(b, p, 1 - p); c = c + (1 - c).detach()         # confianza con straight-through (vale 1 en forward)
        out = c.unsqueeze(-1) * zt + self.res(xh)
        logits = self.head(self.ln(self.dec(out, mask=mascara_causal(T, dev), is_causal=True)))
        return logits, {"p": p, "b": b, "F": b.float().mean(), "G": p.mean(), "chunks": Lc}

def perdida_ratio(F_, G, N):
    """Ratio loss de H-Net (Ec. 10): minimiza ~1 cuando F = G = 1/N; el gradiente fluye por G."""
    return N / (N - 1) * ((N - 1) * F_ * G + (1 - F_) * (1 - G))

# prueba de causalidad (mismo test que en el módulo 06)
_m = MiniHNet().eval(); _x = torch.randint(0, 256, (2, 64)); _x2 = _x.clone(); _x2[:, 40] = (_x2[:, 40] + 1) % 256
with torch.no_grad():
    _d = (_m(_x)[0] - _m(_x2)[0]).abs().amax(-1)
print("máx. diferencia en posiciones < 40:", float(_d[:, :40].max()), "(debe ser 0) | en >= 40:", float(_d[:, 40:].max()))

### 5.1 Entrenamiento comparado: chunking aprendido vs espacios vs *pool* fijo

Tres variantes **idénticas salvo la función de chunking** (como en la Tabla 1 del paper): `dc` (aprendido, con ratio loss con $N=4$), `space` (inicios de palabra, heurística de SpaceByte) y `pool` (cada 4 bytes). Mismo presupuesto de pasos y datos. Una sola semilla: **mira tendencias, no diferencias de milésimas**.

In [ ]:
PASOS_H = 400 if FAST_DEV_RUN else 4000
N_RATIO, ALPHA, T_B, B_TR = 4.0, 0.03, 128, 16

def entrena_hnet(modo, pasos=PASOS_H, seed=SEED, snaps=(0, 100, 250, 400)):
    torch.manual_seed(seed)
    m = MiniHNet(modo=modo, d=64 if FAST_DEV_RUN else 256, d_main=128 if FAST_DEV_RUN else 512, Lm=3 if FAST_DEV_RUN else 8).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-3); g = torch.Generator().manual_seed(seed)
    gv = torch.Generator().manual_seed(1); xv, yv = lote(VA, 64, T_B, gv)
    es_txt = next((d["text"] for d in DOCS_VA if d["lang"] == "es"), DOCS_VA[0]["text"]).strip()[:T_B].ljust(T_B).encode()[:T_B]
    xs = torch.from_numpy(np.frombuffer(es_txt, np.uint8).astype(np.int64))[None].to(device)   # ventana fija en español
    log = {"bpb": [], "F": [], "G": [], "snap": {}, "paso_bpb": []}; t0 = time.time()
    for s in range(pasos + 1):
        if s % 50 == 0 or s == pasos:
            m.eval()
            with torch.no_grad():
                lg, info = m(xv); log["bpb"].append(F.cross_entropy(lg.reshape(-1, 256), yv.reshape(-1)).item() / math.log(2)); log["paso_bpb"].append(s)
            m.train()
        if s in snaps:
            m.eval()
            with torch.no_grad(): log["snap"][s] = (m(xs)[1]["p"][0].cpu().numpy(), xs[0].cpu().numpy())
            m.train()
        if s == pasos: break
        x, y = lote(TR, B_TR, T_B, g); lg, info = m(x)
        loss = F.cross_entropy(lg.reshape(-1, 256), y.reshape(-1))
        if modo == "dc": loss = loss + ALPHA * perdida_ratio(info["F"], info["G"], N_RATIO)
        opt.zero_grad(); loss.backward(); opt.step()
        log["F"].append(float(info["F"])); log["G"].append(float(info["G"]))
    m.eval(); log["modelo"] = m
    print(f"H-Net[{modo:5s}] params={sum(p.numel() for p in m.parameters())/1e6:.2f}M | {time.time()-t0:4.0f}s | bpb final={log['bpb'][-1]:.3f}"
          + (f" | F={np.mean(log['F'][-20:]):.3f} (objetivo {1/N_RATIO:.3f})" if modo == "dc" else ""))
    return log

RUNS = {modo: entrena_hnet(modo) for modo in ["dc", "space", "pool"]}

In [ ]:
cols = {"dc": AZUL, "space": VERDE, "pool": GRIS}
nom = {"dc": "H-Net (chunking aprendido)", "space": "H-Net (espacios, heurística)", "pool": "H-Net (pool fijo k=4)"}
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for k, r in RUNS.items():
    axs[0].plot(r["paso_bpb"], r["bpb"], color=cols[k], lw=2, label=nom[k])
axs[0].set_xlabel("paso"); axs[0].set_ylabel("bpb (validación)"); axs[0].set_title("Calidad: bits por byte"); axs[0].legend(fontsize=8)
ss = lambda v, w=10: np.convolve(v, np.ones(w) / w, mode="valid")
axs[1].plot(ss(RUNS["dc"]["F"]), color=AZUL, lw=2, label="F = fracción de bordes (no diferenciable)")
axs[1].plot(ss(RUNS["dc"]["G"]), color=NARANJA, lw=2, label="G = prob. media de borde (diferenciable)")
axs[1].axhline(1 / N_RATIO, color="crimson", ls="--", label=f"objetivo 1/N = {1/N_RATIO:.2f}"); axs[1].set_xlabel("paso"); axs[1].set_title("La ratio loss lleva F y G hacia 1/N"); axs[1].legend(fontsize=8)
for k in ["dc", "space"]:
    mm = RUNS[k]["modelo"]; xx, _ = lote(VA, 64, T_B, torch.Generator().manual_seed(3))
    with torch.no_grad(): inf_ = mm(xx)[1]
    b_ = inf_["b"].cpu().numpy(); w_ = inicios_espacio_t(xx).cpu().numpy()
    L_ = np.concatenate([np.diff(np.r_[np.flatnonzero(r_), len(r_)]) for r_ in b_])
    axs[2].hist(np.clip(L_, 1, 25), bins=np.arange(.5, 26.5, 1), alpha=.55, color=cols[k], label=f"{nom[k]} (media {L_.mean():.1f})", density=True)
axs[2].set_xlabel("bytes por chunk"); axs[2].set_title("Longitudes de chunk (val)"); axs[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * **Centro:** $F$ (fracción real de bordes) se acerca al objetivo $1/N$ gracias a la ratio loss: el router **no colapsa** ni a «todo» ni a «nada». $G$ (probabilidad media, por la que fluye el gradiente) tarda más y suele quedar por encima: con un entrenamiento tan corto el router aún es poco «confiado» (muchas $p_t$ cerca de 0.5).
> * **Izquierda:** en el paper el orden es *pool fijo < espacios < chunking aprendido*. A esta escala (2 min de CPU, una semilla) las diferencias pueden ser pequeñas o invertirse: lo que valida el mecanismo es que **aprende** y que el coste es el de ~1/4 de los pasos de la red principal.
> * **Derecha:** las longitudes de chunk aprendidas son **variables** (como las de BPE o entropía) mientras `pool` tiene una sola longitud.

In [ ]:
fig, axs = plt.subplots(len(RUNS["dc"]["snap"]), 1, figsize=(12, 1.9 * len(RUNS["dc"]["snap"])), sharex=True)
for ax, (s, (p, xs_)) in zip(axs, RUNS["dc"]["snap"].items()):
    w_ = inicios_espacio_t(torch.from_numpy(xs_)[None])[0].numpy()
    ax.bar(range(len(p)), p, color=[NARANJA if v >= .5 else AZUL for v in p], width=.85)
    for i in np.flatnonzero(w_): ax.axvline(i, color="green", lw=.7, alpha=.5)
    ax.axhline(.5, color="crimson", ls="--", lw=.8); ax.set_ylabel(f"paso {s}\np_t", fontsize=8); ax.set_ylim(0, 1.05)
axs[-1].set_xticks(range(len(xs_))); axs[-1].set_xticklabels([("␣" if c == 32 else chr(c) if 32 < c < 127 else "·") for c in xs_], fontsize=5)
axs[0].set_title("Probabilidad de frontera p_t (naranja = borde; líneas verdes = inicio de palabra real) durante el entrenamiento", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

mm = RUNS["dc"]["modelo"]; xx, _ = lote(VA, 128, T_B, torch.Generator().manual_seed(3))
with torch.no_grad(): b_ = mm(xx)[1]["b"].cpu().numpy()
w_ = inicios_espacio_t(xx).cpu().numpy(); tp = (b_ & w_).sum()
print(f"fronteras aprendidas vs inicios de palabra reales -> precisión={tp/max(1,b_.sum()):.2f} recall={tp/max(1,w_.sum()):.2f} "
      f"(azar con la misma densidad: precisión≈{w_.mean():.2f}, recall≈{b_.mean():.2f})")

> **👀 Qué debes observar**
>
> Si el entrenamiento ha avanzado lo suficiente, las barras naranjas (bordes) empiezan a **coincidir con las líneas verdes (inicios de palabra)** sin que se haya dado ninguna supervisión de espacios. Con 400 pasos el efecto es parcial: compara la **precisión** y el **recall** impresos con los del azar (misma densidad de bordes). Los autores del paper reportan fronteras mucho más claras (palabras, morfemas) tras entrenar con ~100B de tokens.

> ### 🎬 GEMINI-VIDEO · ID: `M07-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Cómo aprende H-Net dónde cortar**
>
> **Qué muestra:** Evolución (ilustrativa) de la probabilidad de frontera p_t a lo largo del entrenamiento: de ruido a picos en inicios de palabra, con la ratio loss estabilizando la densidad de bordes.
> **Formato:** MP4 16:9, 1080p | Duración: 28 s | **Texto alternativo:** Animación de barras de probabilidad de frontera que se concentran en los inicios de palabra a lo largo del entrenamiento
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Video explicativo técnico de 28 segundos, 16:9, 1080p, ritmo calmado, sin voz en off. Estilo: diagrama limpio sobre fondo blanco, tipografía sans-serif legible, paleta azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Sin ilustraciones decorativas ni datos reales (todo es ilustrativo y debe decirlo una nota fija abajo: "Ilustración conceptual").
> Escena fija: la frase "el servidor procesa las peticiones" en una fila de casillas (una por carácter, espacios como ␣) abajo; encima, una fila de barras azules que representan p_t (probabilidad de frontera) y una línea roja discontinua en 0.5. Un contador arriba a la derecha dice "paso de entrenamiento".
> Guion por escenas:
> 0–5 s: título "H-Net aprende sus propias fronteras". Aparece la frase y las barras con alturas casi aleatorias (entre 0.3 y 0.7); contador "paso 0". Etiqueta: "al inicio, el router no sabe nada".
> 5–14 s: el contador avanza a "paso 100" y "paso 250"; las barras se van diferenciando: las que están sobre la primera letra de cada palabra (e, s, p, l, p) suben por encima de 0.5 y se vuelven naranjas; las demás bajan. Aparecen líneas verticales verdes tenues en los inicios de palabra reales para comparar.
> 14–22 s: contador "paso 400". Aparece una barra lateral pequeña con dos marcadores: "F (bordes reales)" y "G (prob. media)" convergiendo hacia una línea horizontal etiquetada "objetivo 1/N". Texto: "la ratio loss mantiene la densidad de bordes".
> 22–28 s: texto final: "Las fronteras emergen sin supervisión; el router decide qué vectores pasan a la red principal". Se desvanece.
> Texto en pantalla en español, con términos técnicos en inglés (router, ratio loss).
> Evita: voz en off, texto ilegible, datos numéricos distintos de los indicados, cambios bruscos, logotipos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M07-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M07-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> paso 0  : barras ~0.5 al azar sobre "el servidor procesa las peticiones"
> paso 250: barras altas (naranja) sobre e, s, p, l, p; bajas en el resto
> paso 400: F y G convergen a 1/N  ->  bordes ~ inicios de palabra
> ```

> **👀 Qué debes observar**
>
> Observa dos cosas: (1) las fronteras **emergen** del objetivo de predecir el siguiente byte (no hay etiqueta de «inicio de palabra»); (2) la ratio loss actúa como **termostato** de la densidad de bordes: sin él, el router podría dar bordes en todos los bytes y anular el ahorro.

## 6. ¿Está muerta la tokenización?

El argumento «los tokenizers desaparecerán» es atractivo; el argumento «es una moda» también. Pongamos la evidencia lado a lado (todas las cifras son **reportadas por los autores**, a oct-2026).

### Evidencia a favor de los modelos sin tokenizer

| Evidencia | Fuente | Matiz |
|---|---|---|
| Un modelo de bytes con parches por entropía iguala a Llama 3 a 8B (1T tokens / ~4.5T bytes, mismos FLOPs) y gana en ortografía (CUTE 54.1 vs 27.5) | BLT, arXiv:2412.09871 | Un solo conjunto de datos y receta; BPE gana a bajo cómputo |
| H-Net de 1 etapa sobre bytes supera a un Transformer BPE (GPT-2) con FLOPs y datos igualados; 2 etapas ≈ un Transformer del doble de tamaño; ventaja mayor en chino, código y ADN | H-Net, arXiv:2507.07955 | Escala ≤ 1.3B; baseline = tokenizer GPT-2 (50K), más débil que los de Llama 3 o similares |
| Convertir (*byte-ify*) un LLM subword en byte-level con poco entrenamiento adicional es viable; Bolmo 1B/7B abiertos | Bolmo, arXiv:2512.15586 | Parte de un backbone entrenado **con** tokenizer: hereda su conocimiento |
| Jerarquías bytes→palabras aprendidas son competitivas con BPE | AU-Net, arXiv:2506.14761 (NeurIPS 2025) | «Jerarquías poco profundas»; las profundas son «prometedoras» |

### Evidencia en contra (o que matiza)

| Evidencia | Fuente | Matiz |
|---|---|---|
| Hasta ~2x más lento en entrenamiento y difícil de *batchear* en inferencia (cómputo dinámico) | H-Net, §Discusión | «No fundamental», dicen; pero hoy es un coste real |
| FLOPs ≠ latencia: las implementaciones actuales pueden no igualar el reloj de los transformers tokenizados | BLT, §9 | |
| Los tokenizers siguen mejorando: **SuperBPE** (tokens que cruzan espacios) da +4.0 % medio en 30 tareas con 8B y hasta 33 % menos tokens | Liu et al., arXiv:2503.13423 | Mejor BPE, no «sin BPE» |
| La compresión no es lo único: PathPiece (mínimo nº de tokens) **no** da mejor rendimiento | Schmidt et al., EMNLP 2024 | Matiza el mito «menos tokens = mejor» |
| Ecosistema: caché de prefijos por token, *constrained decoding*, facturación, evaluación y herramientas de interpretabilidad asumen tokens | — | Coste de migración, no científico |
| No hay (que yo sepa) evidencia pública de un LLM de frontera de producción sin tokenizer | — | «Sin evidencia pública» ≠ «no existe» |

**Veredicto de este curso (opinión razonada, no hecho):** la tokenización **no está muerta, se está internalizando**. Hay buenas razones científicas para que la segmentación se aprenda (equidad multilingüe, robustez, dominios sin vocabulario natural como ADN), pero a oct-2026 los resultados llegan a ≈ 1-8B y la ingeniería sigue por detrás. Un cambio de opinión razonable se produciría con (1) replicación independiente a > 30B, (2) parity de latencia real en *serving*, o (3) un modelo de frontera que lo declare.

In [ ]:
# Cifras REPORTADAS por Hwang, Wang y Gu (2025), arXiv:2507.07955, Tabla 2 (FLOPs igualados a GPT-3 Large 760M / XL 1.3B, 100B tokens de FineWeb-Edu)
modelos = ["Transformer\n(BPE GPT-2)", "SpaceByte++", "H-Net\n(espacios)", "H-Net\n1 etapa", "H-Net\n2 etapas"]
bpb_L = [0.756, 0.760, 0.755, 0.755, 0.743]; acc_L = [53.3, 53.6, 53.4, 53.6, 55.5]
bpb_XL = [0.730, 0.733, 0.726, 0.728, 0.715]; acc_XL = [55.5, 56.1, 57.1, 56.2, 58.2]
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8)); x = np.arange(5)
axs[0].bar(x - .2, bpb_L, .4, color=NARANJA, label="Large (760M FLOPs)"); axs[0].bar(x + .2, bpb_XL, .4, color=AZUL, label="XL (1.3B FLOPs)")
axs[0].set_ylim(.68, .78); axs[0].set_xticks(x); axs[0].set_xticklabels(modelos, fontsize=8); axs[0].set_ylabel("bpb FineWeb-Edu (menor = mejor)"); axs[0].legend(); axs[0].set_title("Bits por byte (reportado)")
axs[1].bar(x - .2, acc_L, .4, color=NARANJA); axs[1].bar(x + .2, acc_XL, .4, color=AZUL)
axs[1].set_ylim(50, 60); axs[1].set_xticks(x); axs[1].set_xticklabels(modelos, fontsize=8); axs[1].set_ylabel("media de 7 tareas zero-shot (%)"); axs[1].set_title("Precisión media (reportado)")
plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * Las barras son **cifras de las tablas del paper** (no ejecutadas aquí). H-Net 2 etapas mejora 0.013-0.015 bpb sobre el Transformer BPE en ambas escalas; con tareas zero-shot, +2.2 y +2.6 puntos de media según el paper.
> * Mira los **ejes truncados**: las diferencias de bpb entre H-Net 1 etapa, espacios y Transformer son del orden de 0.001-0.005. Los autores mismos hablan de «ruido» en algunas comparaciones de evaluación.
> * La conclusión robusta del paper es *cualitativa*: la jerarquía y el chunking aprendido **se pagan solos** a FLOPs iguales y su ventaja crece con los datos (cruce con el Transformer a los 30B bytes para 2 etapas en XL).

## 7. Tokenizer transfer: cambiar el vocabulario de un modelo ya entrenado

**Problema (Minixhofer et al., 2024).** Un LM está *atado* a su tokenizer: su matriz de embeddings $E_a\in\mathbb{R}^{|V_a|\times d}$ (y, si no hay *weight tying*, la de salida) solo tiene sentido para $V_a$. Queremos un tokenizer nuevo $(V_b, T_b)$ —otro idioma, dominio o tamaño— **conservando el cuerpo del modelo**. Todo se reduce a: *¿cómo inicializar $E_b$?* Después, un ajuste corto.

### 7.1 Taxonomía de métodos (lo que necesitan y cómo inicializan)

| Método | Idea | Necesita | Verificado en |
|---|---|---|---|
| Aleatoria | $e_b(t)\sim\mathcal{N}(0,\sigma^2)$ | nada | línea base |
| Media / Hewitt | todos los nuevos = media de $E_a$ (o muestra de $\mathcal{N}(\mu,\Sigma)$ de $E_a$) | nada | `transformers` (`mean_resizing=True`) |
| **FVT** | $e_b(t)=\frac1{\lvert T_a(t)\rvert}\sum_{s\in T_a(t)} e_a(s)$: media de la descomposición con el tokenizer antiguo | tokenizer antiguo | Gee et al. 2022; descrito en ZeTT §2 |
| **WECHSEL** | embeddings estáticos (fastText) de ambos idiomas alineados con diccionario bilingüe + Procrustes; $e_b(t)=\sum_{s\in \text{kNN}}\mathrm{softmax}(\cos/\tau)\,e_a(s)$, $\tau=0.1$, $k=10$ | fastText ambos idiomas + diccionario | arXiv:2112.06598 |
| **FOCUS** | copia los tokens **compartidos**; cada nuevo = combinación **sparsemax** de los compartidos, con pesos por similitud en un espacio auxiliar (fastText sobre el idioma destino) | datos no etiquetados del destino | arXiv:2305.14481 |
| **Trans-Tokenization** | alinea tokens de origen y destino con un **corpus paralelo** (alineador estadístico) y promedia ponderando por recuento de alineación | corpus paralelo (o diccionario) | arXiv:2408.04303 (COLM 2024) |
| **ZeTT** | una **hiperred** (otro LM pequeño) que, dado un tokenizer, *predice* sus embeddings componiendo los de la descomposición; se entrena sobre muchos tokenizers y generaliza a uno nuevo **sin datos del destino** | entrenar la hiperred | arXiv:2405.07883 (NeurIPS 2024) |

**Ejemplo a mano (FVT).** El tokenizer nuevo tiene `▁servidor`; el antiguo lo parte en `▁serv` + `idor` con embeddings (de juguete, $d=2$) $(1,0)$ y $(0,2)$. FVT lo inicializa con la media: $e_b(\texttt{▁servidor})=(0.5,\,1)$. Un token que el antiguo ya tenía igual se copia tal cual. Es la misma intuición que el *cold start* de un ítem nuevo en recsys: lo inicializas con la media de los ítems con los que se relaciona, no al azar.

**Resultados reportados por ZeTT** (no los hemos reproducido): con XLM-R, tokenizers específicos por idioma acortan las secuencias ≈ 14 % en media, aceleran la inferencia > 16 % y conservan XNLI a ≈ 1 % en promedio; con Mistral-7B, un tokenizer de código con ~10 % menos tokens preserva la corrección funcional a ≈ 3 %; y el hueco restante se cierra con **< 1B tokens** de entrenamiento continuado, frente a cientos de miles de millones en trabajos previos. En decodificadores el problema es más difícil: FOCUS «es aproximadamente aleatorio en el peor caso» (−23.2 % en BoolQ) y da 0 % pass@1 en HumanEval Python.

**Otros caminos:** OFA (parametrización factorizada de embeddings, arXiv:2311.08849), AdaptiVocab (adaptar el vocabulario a un dominio, arXiv:2503.19693), *retrofitting* con tokenización dinámica (arXiv:2411.18553) y destilación entre tokenizers (ALM, arXiv:2503.20083).

### 7.2 La métrica correcta
Tras el cambio, la pérdida *por token* **no es comparable** (los tokens nuevos cubren más bytes). Evaluamos **bits por byte** sobre el mismo texto con cada tokenizer.

### 7.3 Montaje del experimento

Modelo base: **GPT-2 en miniatura** (`GPT2LMHeadModel`, 3 capas, ancho 128) preentrenado solo con **inglés + código** y un BPE de 512 tokens (tokenizer A). Destino: **español** con un BPE propio de 768 tokens (tokenizer B). En el modo completo se usa el **GPT-2 small real** (124M) y su tokenizer de 50 257 tokens si tienes acceso al hub.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

def entrena_bpe(docs_, vocab_size):
    tk = Tokenizer(models.BPE())
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    trn = trainers.BpeTrainer(vocab_size=vocab_size, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
                              show_progress=False)
    tk.train_from_iterator((d["text"][:MAX_CHARS] for d in docs_), trn)
    return tk

def bytes_por_token(tk, textos):
    n_b = sum(len(t.encode()) for t in textos)
    return n_b / sum(len(tk.encode(t).ids) for t in textos)

In [ ]:
def bytes_to_unicode():
    """Mapa GPT-2 byte -> carácter imprimible (el que usa ByteLevel). 1 carácter == 1 byte en el vocabulario."""
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)); cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
B2U = bytes_to_unicode(); U2B = {v: k for k, v in B2U.items()}
tok_a_bytes = lambda s: bytes(U2B[c] for c in s)

from tokenizers import Tokenizer
SRC = [d for d in DOCS_TR if d["lang"] in ("en", "code")]            # "idioma de origen" del modelo base
TGT = [d for d in DOCS_TR if d["lang"] == "es"]                       # idioma destino de la adaptación
TGT_VA = [d for d in DOCS_VA if d["lang"] == "es"]
V_A, V_B = (512, 768) if FAST_DEV_RUN else (50257, 24000)
TK_A = entrena_bpe(SRC, V_A)                                          # tokenizer original (inglés + código)
TK_B = entrena_bpe(TGT, V_B)                                          # tokenizer nuevo (español)
if not FAST_DEV_RUN:
    try:
        TK_A = Tokenizer.from_pretrained("gpt2")                     # modo completo: GPT-2 de verdad (requiere acceso a Hugging Face)
        V_A = TK_A.get_vocab_size()
    except Exception as e:
        print("(sin acceso al hub; sigo con el BPE entrenado en local)", type(e).__name__)

VA_ES = "\n\n".join(d["text"][:MAX_CHARS] for d in TGT_VA)
def bytes_x_token(tk, txt): return len(txt.encode()) / len(tk.encode(txt).ids)
print(f"vocab A={TK_A.get_vocab_size()} | vocab B={TK_B.get_vocab_size()}")
print(f"bytes/token en ES (val): tokenizer A = {bytes_x_token(TK_A, VA_ES):.2f}  ->  tokenizer B = {bytes_x_token(TK_B, VA_ES):.2f}")
solap = set(TK_A.get_vocab()) & set(TK_B.get_vocab())
print(f"tokens de B que ya existen en A (se copian): {len(solap)} de {TK_B.get_vocab_size()} ({100*len(solap)/TK_B.get_vocab_size():.0f} %)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6)); langs = ["es", "en", "code"]
for k, (tk, nombre, col) in enumerate([(TK_A, "tokenizer A (EN+código)", NARANJA), (TK_B, "tokenizer B (ES)", AZUL)]):
    vals = [np.mean([bytes_x_token(tk, d["text"][:MAX_CHARS]) for d in DOCS_VA if d["lang"] == l]) for l in langs]
    ax.bar(np.arange(3) + (k - .5) * .38, vals, .38, color=col, label=nombre)
ax.set_xticks(range(3)); ax.set_xticklabels(langs); ax.set_ylabel("bytes por token (más = menos tokens)"); ax.set_title("Lo que ganas al cambiar de tokenizer"); ax.legend(); plt.show()

> **👀 Qué debes observar**
>
> El tokenizer B (entrenado en español) produce tokens más largos sobre el español: **menos pasos del transformer por byte**. Ese es el beneficio de adaptar el vocabulario; el coste es el de inicializar y ajustar el nuevo $E_b$. (En el módulo 04 viste lo mismo con vocabularios reales: la «tarifa por idioma».)

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel
T_TOK = 64

def ids_stream(tk, docs_):
    return np.array(tk.encode("\n\n".join(d["text"][:MAX_CHARS] for d in docs_)).ids, dtype=np.int64)

def lote_ids(ids, B, T, g):
    ix = torch.randint(0, len(ids) - T - 1, (B,), generator=g)
    x = torch.stack([torch.from_numpy(ids[i:i + T]) for i in ix]); y = torch.stack([torch.from_numpy(ids[i + 1:i + T + 1]) for i in ix])
    return x.to(device), y.to(device)

def largo_bytes(tk):                                                   # bytes de cada token (en ByteLevel 1 carácter = 1 byte)
    return np.array([len(tok_a_bytes(t)) if all(c in U2B for c in t) else 1 for t, _ in sorted(tk.get_vocab().items(), key=lambda kv: kv[1])])

@torch.no_grad()
def bpb(model, tk, txt, T=T_TOK):
    """bits por byte sobre un texto (ventanas contiguas sin solape): la métrica comparable entre tokenizers."""
    ids = np.array(tk.encode(txt).ids, dtype=np.int64); LB = largo_bytes(tk); model.eval(); nll = nb = 0.0
    for i in range(0, len(ids) - T - 1, T):
        x = torch.from_numpy(ids[i:i + T])[None].to(device); y = torch.from_numpy(ids[i + 1:i + T + 1])[None].to(device)
        nll += F.cross_entropy(model(input_ids=x).logits[0], y[0], reduction="sum").item(); nb += LB[ids[i + 1:i + T + 1]].sum()
    return nll / math.log(2) / nb

def nuevo_gpt(vocab, d=128, L=3, H=4):
    return GPT2LMHeadModel(GPT2Config(vocab_size=vocab, n_positions=T_TOK, n_embd=d, n_layer=L, n_head=H,
                                      resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0, bos_token_id=0, eos_token_id=0)).to(device)

def entrena_lm(model, ids, pasos, lr=2e-3, params=None, log_cada=None, evalfn=None, seed=SEED):
    opt = torch.optim.AdamW(params if params is not None else model.parameters(), lr=lr); g = torch.Generator().manual_seed(seed); curva = []
    for s in range(pasos + 1):
        if log_cada and s % log_cada == 0: curva.append((s, evalfn(model)))
        if s == pasos: break
        model.train(); x, y = lote_ids(ids, 16, T_TOK, g)
        loss = F.cross_entropy(model(input_ids=x).logits.reshape(-1, model.config.vocab_size), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return curva

# --- Modelo base: preentrenado SOLO con inglés + código y el tokenizer A ---
t0 = time.time(); torch.manual_seed(SEED)
IDS_A_SRC = ids_stream(TK_A, SRC)
if FAST_DEV_RUN or TK_A.get_vocab_size() != 50257:
    BASE = nuevo_gpt(TK_A.get_vocab_size()); entrena_lm(BASE, IDS_A_SRC, 500 if FAST_DEV_RUN else 3000)
else:
    BASE = GPT2LMHeadModel.from_pretrained("gpt2").to(device)         # modo completo: GPT-2 small preentrenado (124M)
VA_SRC = "\n\n".join(d["text"][:MAX_CHARS] for d in DOCS_VA if d["lang"] in ("en", "code"))
print(f"modelo base listo en {time.time()-t0:.0f}s | bpb(val EN+código, tokenizer A) = {bpb(BASE, TK_A, VA_SRC):.3f}")
print(f"bpb(val ES, tokenizer A) ANTES de adaptar = {bpb(BASE, TK_A, VA_ES):.3f}   <- el modelo no ha visto español")

### 7.4 Inicializaciones desde cero

Seis maneras de rellenar $E_b$. Todas **copian** los tokens cuya cadena aparece en ambos vocabularios (los «compartidos») y difieren en cómo tratan al resto.

* **Hewitt:** muestrear de $\mathcal{N}(\mu,\Sigma)$ ajustada a $E_a$ para que el softmax no reparta masa extra a los nuevos. (La implementación por defecto de Hugging Face usa $\epsilon\Sigma$ con $\epsilon=10^{-9}$: en la práctica, la media.)
* **FOCUS-lite:** `sparsemax` sobre similitudes en un espacio auxiliar. Aquí el espacio auxiliar es **PPMI + SVD** sobre el texto tokenizado (un «fastText de juguete»); el paper usa fastText.
* **WECHSEL-lite:** espacios auxiliares de A y B **alineados con Procrustes** usando los tokens compartidos como «diccionario», y media ponderada por softmax (τ=0.1, k=10).

In [ ]:
def sparsemax(z):
    """Martins & Astudillo (2016): proyección euclídea al simplex; produce pesos EXACTAMENTE cero fuera del soporte."""
    zs, _ = torch.sort(z, descending=True); k = torch.arange(1, len(z) + 1, device=z.device)
    cs = zs.cumsum(0); sop = (1 + k * zs) > cs; kz = int(sop.sum()); tau = (cs[kz - 1] - 1) / kz
    return torch.clamp(z - tau, min=0)

def vocab_ordenado(tk): return [t for t, _ in sorted(tk.get_vocab().items(), key=lambda kv: kv[1])]

def descomposicion(tk_old, token_str):
    """Ids del tokenizer ANTIGUO que componen el texto de un token nuevo (FVT/ZeTT). Si no es UTF-8 válido, usa bytes sueltos."""
    try:
        txt = tok_a_bytes(token_str).decode("utf-8")
        ids = tk_old.encode(txt).ids
        if ids: return ids
    except Exception:
        pass
    v = tk_old.get_vocab(); return [v[c] for c in token_str if c in v]

def base_copia(E_A, tkA, tkB, std=0.02):
    """Matriz [V_B, d] con los tokens compartidos copiados (cadena idéntica) y el resto sin inicializar (ceros)."""
    vA, VB = tkA.get_vocab(), vocab_ordenado(tkB); E = torch.zeros(len(VB), E_A.shape[1]); nuevo = []
    for i, t in enumerate(VB):
        if t in vA: E[i] = E_A[vA[t]]
        else: nuevo.append(i)
    return E, nuevo

def init_aleatoria(E_A, tkA, tkB):
    E, nuevo = base_copia(E_A, tkA, tkB); E[nuevo] = torch.randn(len(nuevo), E_A.shape[1]) * E_A.std(); return E

def init_media(E_A, tkA, tkB):                                         # todos los nuevos = media de los viejos
    E, nuevo = base_copia(E_A, tkA, tkB); E[nuevo] = E_A.mean(0); return E

def init_hewitt(E_A, tkA, tkB, escala=1.0):                            # muestreo de N(media, escala*covarianza) de los viejos
    E, nuevo = base_copia(E_A, tkA, tkB); mu = E_A.mean(0); C = torch.cov((E_A - mu).T) * escala + 1e-6 * torch.eye(E_A.shape[1])
    E[nuevo] = torch.distributions.MultivariateNormal(mu, C).sample((len(nuevo),)); return E

def init_fvt(E_A, tkA, tkB):                                           # Fast Vocabulary Transfer: media de la descomposición
    E, nuevo = base_copia(E_A, tkA, tkB); VB = vocab_ordenado(tkB)
    for i in nuevo: E[i] = E_A[descomposicion(tkA, VB[i])].mean(0)
    return E

In [ ]:
def aux_embeddings(ids, V, ventana=4, dim=32):
    """Embeddings estáticos baratos (PPMI + SVD) de tokens: el 'fastText de juguete' que usan WECHSEL/FOCUS como espacio auxiliar."""
    C = np.zeros((V, V), np.float32)
    for k in range(1, ventana + 1):
        np.add.at(C, (ids[:-k], ids[k:]), 1.0); np.add.at(C, (ids[k:], ids[:-k]), 1.0)
    pr = C.sum(1, keepdims=True) / C.sum(); pmi = np.log((C / C.sum() + 1e-12) / (pr @ pr.T + 1e-12)); ppmi = np.maximum(pmi, 0)
    U, S, _ = np.linalg.svd(ppmi, full_matrices=False); W = U[:, :dim] * np.sqrt(S[:dim])
    return torch.from_numpy(W / (np.linalg.norm(W, axis=1, keepdims=True) + 1e-8)).float()

def init_focus(E_A, tkA, tkB, aux_B):
    """FOCUS (Dobler & de Melo, 2023): cada token NUEVO = combinación sparsemax de los tokens COMPARTIDOS, con pesos por similitud
    coseno en un espacio auxiliar entrenado sobre el idioma destino."""
    E, nuevo = base_copia(E_A, tkA, tkB); vA, VB = tkA.get_vocab(), vocab_ordenado(tkB)
    comp = [i for i, t in enumerate(VB) if t in vA]; Ec = E[comp]; Ac = aux_B[comp]
    for i in nuevo:
        w = sparsemax(Ac @ aux_B[i]); E[i] = (w[:, None] * Ec).sum(0) if w.sum() > 0 else E_A.mean(0)
    return E

def init_wechsel(E_A, tkA, tkB, aux_A, aux_B, k=10, tau=0.1):
    """WECHSEL (Minixhofer et al., 2022): espacio auxiliar de A y de B alineado con Procrustes (diccionario = tokens compartidos);
    cada token nuevo = media ponderada (softmax(cos/tau)) de los k tokens de A más parecidos."""
    from scipy.linalg import orthogonal_procrustes
    E, nuevo = base_copia(E_A, tkA, tkB); vA, VB = tkA.get_vocab(), vocab_ordenado(tkB)
    par = [(vA[t], i) for i, t in enumerate(VB) if t in vA]
    R, _ = orthogonal_procrustes(aux_B[[i for _, i in par]].numpy(), aux_A[[a for a, _ in par]].numpy()); R = torch.from_numpy(R).float()
    S = (aux_B[nuevo] @ R) @ aux_A.T                                    # similitud coseno (filas ya normalizadas)
    top = S.topk(k, dim=1); W = F.softmax(top.values / tau, dim=1)
    E[nuevo] = torch.einsum("nk,nkd->nd", W, E_A[top.indices]); return E

In [ ]:
E_A = BASE.get_input_embeddings().weight.detach().cpu().float()
IDS_B_TGT = ids_stream(TK_B, TGT); IDS_A_TGT = ids_stream(TK_A, TGT)
aux_B = aux_embeddings(IDS_B_TGT, TK_B.get_vocab_size()); aux_A = aux_embeddings(IDS_A_SRC, TK_A.get_vocab_size())
torch.manual_seed(SEED)
INITS = {"aleatoria": init_aleatoria(E_A, TK_A, TK_B), "media": init_media(E_A, TK_A, TK_B),
         "Hewitt (N(μ,Σ))": init_hewitt(E_A, TK_A, TK_B), "FVT (media de subtokens)": init_fvt(E_A, TK_A, TK_B),
         "FOCUS-lite": init_focus(E_A, TK_A, TK_B, aux_B), "WECHSEL-lite": init_wechsel(E_A, TK_A, TK_B, aux_A, aux_B)}

def instala_vocab(base, E_new):
    m = copy.deepcopy(base); m.resize_token_embeddings(E_new.shape[0], mean_resizing=False)
    m.get_input_embeddings().weight.data.copy_(E_new.to(device)); m.tie_weights(); return m

K1, K2 = (100, 200) if FAST_DEV_RUN else (500, 2000)
eval_B = lambda m: bpb(m, TK_B, VA_ES)
CURVAS, CEROSHOT = {}, {}
t0 = time.time()
for nombre, E_new in INITS.items():
    m = instala_vocab(BASE, E_new); CEROSHOT[nombre] = eval_B(m)
    for p_ in m.parameters(): p_.requires_grad = False
    m.get_input_embeddings().weight.requires_grad = True              # fase 1: SOLO embeddings (la cabeza está atada)
    c1 = entrena_lm(m, IDS_B_TGT, K1, lr=3e-3, params=[m.get_input_embeddings().weight], log_cada=25, evalfn=eval_B)
    for p_ in m.parameters(): p_.requires_grad = True                  # fase 2: todo el modelo
    c2 = entrena_lm(m, IDS_B_TGT, K2, lr=1e-3, log_cada=25, evalfn=eval_B)
    CURVAS[nombre] = c1[:-1] + [(s + K1, v) for s, v in c2]
    print(f"{nombre:26s} bpb 0-shot={CEROSHOT[nombre]:.3f} | tras {K1} pasos (emb) = {c1[-1][1]:.3f} | final = {c2[-1][1]:.3f}")
print(f"({time.time()-t0:.0f}s)")

In [ ]:
# Controles: (a) seguir con el tokenizer A (solo ajuste fino en español) y (b) entrenar desde cero con el tokenizer B
t0 = time.time(); ev_A = lambda m: bpb(m, TK_A, VA_ES)
mA = copy.deepcopy(BASE); cA = entrena_lm(mA, IDS_A_TGT, K1 + K2, lr=1e-3, log_cada=25, evalfn=ev_A)
torch.manual_seed(SEED); mS = nuevo_gpt(TK_B.get_vocab_size()); cS = entrena_lm(mS, IDS_B_TGT, K1 + K2, lr=2e-3, log_cada=25, evalfn=eval_B)
CURVAS["control: seguir con A"] = cA; CURVAS["control: desde cero con B"] = cS
print(f"control A: {cA[0][1]:.3f} -> {cA[-1][1]:.3f} | desde cero con B: {cS[0][1]:.3f} -> {cS[-1][1]:.3f} ({time.time()-t0:.0f}s)")

In [ ]:
nombres = list(INITS)
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
bars = axs[0].bar(range(len(nombres)), [CEROSHOT[n] for n in nombres], color=[GRIS, GRIS, GRIS, NARANJA, AZUL, VERDE])
axs[0].axhline(bpb(BASE, TK_A, VA_ES), color="crimson", ls="--", label="base con tokenizer A (no adaptado)")
axs[0].set_xticks(range(len(nombres))); axs[0].set_xticklabels([n.replace(" (", "\n(") for n in nombres], fontsize=7, rotation=0)
axs[0].set_ylabel("bpb en ES (cero-shot, justo tras cambiar de tokenizer)"); axs[0].set_title("Antes de entrenar: ¿qué init sobrevive?"); axs[0].legend(fontsize=8)
paleta = {"aleatoria": "#9aa0a6", "media": "#6B7280", "Hewitt (N(μ,Σ))": "#b6a37a", "FVT (media de subtokens)": NARANJA, "FOCUS-lite": AZUL, "WECHSEL-lite": VERDE,
          "control: seguir con A": "crimson", "control: desde cero con B": "black"}
for n, c in CURVAS.items():
    axs[1].plot([s for s, _ in c], [v for _, v in c], color=paleta[n], lw=2, ls="--" if n.startswith("control") else "-", label=n)
axs[1].axvline(K1, color="k", lw=.6, ls=":"); axs[1].set_ylim(top=min(axs[1].get_ylim()[1], 6)); axs[1].set_xlabel("pasos (a la izquierda de la línea punteada: solo embeddings)")
axs[1].set_ylabel("bpb en ES (validación)"); axs[1].set_title("Adaptación corta"); axs[1].legend(fontsize=6)
axs[2].bar(range(len(nombres)), [dict(CURVAS[n])[K1] for n in nombres], color=[paleta[n] for n in nombres])
axs[2].set_xticks(range(len(nombres))); axs[2].set_xticklabels([n.replace(" (", "\n(") for n in nombres], fontsize=7)
axs[2].set_title(f"bpb tras {K1} pasos (solo embeddings)"); plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * **Izquierda (cero-shot):** justo tras cambiar de tokenizer, las inicializaciones **informadas** (FVT, FOCUS-lite; en nuestra corrida también WECHSEL-lite) quedan algo por debajo de la media/Hewitt, y todas por debajo de la línea roja (el modelo **sin** adaptar). Las diferencias son pequeñas: en ZeTT los heurísticos pueden estar «a nivel del azar» en decodificadores — aquí el modelo es diminuto, no extrapoles.
> * **Centro y derecha:** tras 100 pasos de **solo embeddings** las curvas se separan (FVT/WECHSEL ganan algo a media/Hewitt); al descongelar todo el modelo las diferencias se **comprimen** casi a cero — la inicialización ahorra *pasos*, no cambia el destino (de ahí que ZeTT mida cuántos tokens hacen falta para cerrar el hueco).
> * **Controles:** *seguir con A* (línea roja discontinua) acaba por encima de todos los cambios de tokenizer a igualdad de pasos —con más tokens por byte— y *desde cero con B* (negra) tarda más: el preentrenamiento sí valía. Tu criterio de ingeniero: ¿el bpb final y el ahorro de tokens compensan el coste de ajustar?
> * ⚠️ Una semilla, ~300 pasos, modelo de ~0.5M parámetros: **cualitativo**.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(12, 7)); P0 = torch.pca_lowrank(E_A - E_A.mean(0), q=2)[2]
vA = TK_A.get_vocab(); VB = vocab_ordenado(TK_B); es_nuevo = np.array([t not in vA for t in VB])
for ax, (n, E) in zip(axs.ravel(), INITS.items()):
    old = ((E_A - E_A.mean(0)) @ P0).numpy(); new = ((E - E_A.mean(0)) @ P0).numpy()
    ax.scatter(old[:, 0], old[:, 1], s=6, color=GRIS, alpha=.4, label="viejos (A)")
    ax.scatter(new[es_nuevo, 0], new[es_nuevo, 1], s=8, color=NARANJA, alpha=.7, label="nuevos (B∖A)")
    ax.set_title(f"{n}\nvar. nuevos / var. viejos = {E[es_nuevo].var(0).sum()/E_A.var(0).sum():.2f}", fontsize=8); ax.legend(fontsize=6)
plt.suptitle("Proyección PCA de los embeddings: dónde coloca cada método a los tokens nuevos", y=1.0); plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> Cada panel muestra los embeddings **viejos** (gris) y los **nuevos** (naranja). `media`: los nuevos colapsan en **un punto** (varianza ≈ 0) → todos reciben el mismo logit, el modelo no puede distinguirlos hasta que el ajuste los separe. `aleatoria`: nubes sin relación con el espacio del modelo. `FVT`/`FOCUS`/`WECHSEL`: los nuevos caen **dentro de la nube de los viejos** con varianza propia — «posiciones razonables» desde el paso 0. La razón entre varianzas (título) es un buen diagnóstico rápido.

> ### 🎨 GEMINI-IMAGEN · ID: `M07-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Estrategias para inicializar el vocabulario nuevo**
>
> **Qué muestra:** Compara cuatro maneras de dar un embedding a un token nuevo (media, FVT, FOCUS/sparsemax, WECHSEL/Procrustes) con un ejemplo concreto.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Cuadrícula de cuatro estrategias de inicialización de un token nuevo con pesos de ejemplo
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, estilo infografía de paper (sans-serif legible, flechas claras, sin ilustraciones decorativas ni texto inventado). Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Formato 16:9.
> Título: "¿Qué embedding le damos al token nuevo «ción»?".
> Contexto a la izquierda, en un recuadro gris: "Vocabulario antiguo A (inglés): …, 'tion', 'ation', 'c', 'i', 'ón' …" y "Vocabulario nuevo B (español): …, 'ción', 'dad', …". Debajo, un cuadro: "Tokens compartidos (se copian): 'a', 'e', 'ion'".
> A la derecha, cuatro mini-diagramas en cuadrícula 2×2, cada uno con un título, una flecha desde "ción (nuevo)" hacia su vector de salida (círculo naranja) y los pesos escritos como números sobre las flechas de entrada:
> 1) "Media (Hewitt/HF)": flechas desde TODOS los embeddings del vocabulario antiguo (nube gris) hacia un solo círculo naranja en el centro de la nube. Nota: "todos los nuevos quedan en el mismo punto".
> 2) "FVT": el texto "ción" se descompone con el tokenizer antiguo en "c" + "i" + "ón"; flechas de pesos iguales (1/3, 1/3, 1/3) hacia el círculo naranja. Nota: "media de la descomposición".
> 3) "FOCUS": flechas desde los tokens compartidos con pesos distintos (0.7, 0.3, 0.0) producidos por "sparsemax"; nota: "pesos exactamente 0 fuera del soporte; similitud en un espacio auxiliar entrenado con texto en español".
> 4) "WECHSEL": dos nubes (inglés y español) alineadas con una flecha curva etiquetada "rotación de Procrustes"; el círculo naranja se calcula con los k vecinos más cercanos y "softmax(cos/τ)".
> Pie en gris: "Después de cualquier init: ajuste corto (primero solo embeddings, luego todo el modelo)".
> Evita: ilustraciones decorativas, texto ilegible, números distintos de los indicados, flechas ambiguas, logotipos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M07-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Cuadrícula de cuatro estrategias de inicialización de un token nuevo con pesos de ejemplo](assets/M07-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> Nuevo: 'ción' (B). Compartidos: a, e, ion.
> 1 Media    : todos los nuevos -> media(E_A)                       (colapsan en un punto)
> 2 FVT      : T_A('ción') = [c, i, ón] -> (e_c + e_i + e_ón)/3
> 3 FOCUS    : sparsemax(cos(aux)) sobre compartidos -> 0.7 e_ion + 0.3 e_a + 0 e_e
> 4 WECHSEL  : Procrustes(aux_B -> aux_A) y softmax(cos/0.1) sobre los 10 vecinos más cercanos
> ```

> **👀 Qué debes observar**
>
> Lee el diagrama de menos a más información: **media** (ninguna) → **FVT** (el propio tokenizer antiguo) → **FOCUS** (datos del destino) → **WECHSEL** (datos de ambos + diccionario). ZeTT sustituye esa heurística por una función **aprendida** de la descomposición.

In [ ]:
# Lo que hace Hugging Face por defecto al ampliar el vocabulario: resize_token_embeddings(mean_resizing=True)
m_hf = copy.deepcopy(BASE); n0 = m_hf.get_input_embeddings().weight.shape[0]
m_hf.resize_token_embeddings(n0 + 50, mean_resizing=True)
W = m_hf.get_input_embeddings().weight.detach().cpu()
old_mu, old_sd = W[:n0].mean(0), W[:n0].std(0).mean()
print("¿los 50 tokens nuevos están en la media de los viejos? distancia media a la media:",
      float((W[n0:] - old_mu).norm(dim=1).mean()), "| norma media de un embedding viejo:", float(W[:n0].norm(dim=1).mean()))
print("=> con mean_resizing=True los nuevos son casi idénticos entre sí (la covarianza se escala por 1e-9 en el código fuente de transformers).")

### 7.5 Checklist para un cambio de tokenizer en un proyecto real

1. **Tokens especiales:** `pad`, `eos`, `bos`, plantillas de chat; si no se mapean, el modelo "no sabe parar".
2. **Cabeza de salida:** con *weight tying* se comparte; si no, hay que inicializar **también** la matriz de salida (ZeTT predice las dos).
3. **Reetiquetar los datos** con el tokenizer nuevo y recalcular longitudes: el contexto efectivo en bytes cambia.
4. **Medir en bpb** (o en una métrica de tarea), con el mismo texto, antes y después; cuidado con filtrar el *test* en los datos de ajuste.
5. **Regresión en lo que no se adaptó:** el inglés y el código no deben degradarse (mide el bpb sobre ellos).
6. **Presupuesto de ajuste:** primero solo embeddings (barato y estable), luego descongelar. Registra cuántos tokens hacen falta para cerrar el hueco.

## 8. Mapa de madurez (a oct-2026)

Con H-Net y tokenizer transfer ya vistos, esta tabla resume qué es sólido, emergente o experimental.

| Técnica | Madurez | Quién la usa / respaldo público | Fuente |
|---|---|---|---|
| BLT (parches por entropía, módulo 06) | 🟡/🔴 | Meta FAIR; hasta 8B, pesos 1B/7B | arXiv:2412.09871 (ACL 2025) |
| **H-Net / dynamic chunking** | 🔴 | CMU / Cartesia (Albert Gu); código `goombalab/hnet` y pesos en la organización `cartesia-ai` de Hugging Face (1 y 2 etapas, tamaños L y XL); ICLR 2026; hasta ~1.3B de FLOPs equivalentes | arXiv:2507.07955 |
| Seguimiento de H-Net (SOMBRERO, H-Net++, chunking adaptativo…) | 🔴 | Grupos académicos / industriales; resultados a ~1B o menos | p. ej. arXiv:2601.22805 |
| AU-Net (U-Net autoregresiva bytes→palabras) | 🔴 | Meta FAIR / NeurIPS 2025; «en pie de igualdad» con BPE en jerarquías poco profundas | arXiv:2506.14761 |
| *Byte-ification* (Bolmo) | 🔴/🟡 | Ai2: Bolmo 1B/7B abiertos, convertidos desde Olmo 2/3 | arXiv:2512.15586 |
| Init por media / muestreo de la distribución (Hewitt) | 🟢 | Es el **valor por defecto** de `resize_token_embeddings(mean_resizing=True)` en `transformers` | Documentación de `transformers`; blog de J. Hewitt (2021) |
| FVT (media de la descomposición) | 🟢/🟡 | Compresión de modelos (Gee et al., 2022); baseline estándar en la literatura de transfer | arXiv:2402.09977 |
| WECHSEL | 🟡 | Transferencia de GPT-2/RoBERTa a otros idiomas; código abierto | arXiv:2112.06598 |
| FOCUS | 🟡 | Especialización monolingüe de XLM-R; checkpoints y código publicados | arXiv:2305.14481 |
| Trans-Tokenization | 🟡 | Familia «Tweeties» (COLM 2024: tatar, neerlandés, armenio, italiano), derivada de Mistral-7B (p. ej. Tweety-Tatar desde Mistral-7B-Instruct-v0.2, según su model card) | arXiv:2408.04303 |
| ZeTT (hiperred) | 🟡/🔴 | NeurIPS 2024; hiperredes publicadas (XLM-R, Mistral-7B, TinyLlama; Llama 3 8B como experimental) | arXiv:2405.07883 |
| Destilación entre tokenizers (ALM) | 🔴 | Investigación (NeurIPS 2025) | arXiv:2503.20083 |

> **Regla del curso:** «🔴» = hay un paper con resultados a pequeña/mediana escala y código, pero **no hay evidencia pública de adopción en producción a gran escala**.

## 9. 🏭 En producción

* **Cambiar de tokenizer es una práctica real, pero pequeña:** se hace para **adaptar un modelo abierto a un idioma o dominio** (p. ej. las familias monolingües derivadas con WECHSEL/FOCUS/Trans-Tokenization y la serie «Tweeties» de Mistral-7B) y para **comprimir** modelos (FVT). Los laboratorios grandes, cuando necesitan un idioma nuevo, a menudo extienden el vocabulario y continúan el preentrenamiento; las cifras concretas rara vez se publican.
* **Hugging Face:** `resize_token_embeddings` inicializa por defecto con la media/covarianza de los embeddings viejos (`mean_resizing=True`). Para añadir tokens de dominio es lo razonable; para **sustituir** el vocabulario entero, no.
* **H-Net en la práctica:** los autores publicaron código (`goombalab/hnet`) y pesos; la propia discusión del paper avisa de entrenamiento hasta ≈2x más lento, de memoria dinámica y de la dificultad de *batchear* la inferencia con coste desigual por posición. Trátalo como **investigación reproducible**, no como dependencia de producción.
* **Políglota:** (1) para el **embedder** (módulos 09-12) la decisión práctica es *qué tokenizer heredas* de un encoder existente y cuándo extenderlo con tokens de dominio (conecta con el 09); (2) para un futuro **generativo**, seguir BLT/H-Net/Bolmo como opción de largo plazo, no de la hoja de ruta actual.

## 10. 🧠 Secretos de la élite

1. **La ratio loss es el 'termostato'.** Sin ella el router colapsa (todo o nada); con ella, $F\to G\to 1/N$. Observa $F$ y $G$ por separado en tus logs: si $F\ne G$ de forma persistente, el router no es confiado (el paper nota que $\mathcal{L}_{\text{ratio}}$ puede bajar de 1 cuando $F\neq G$).
2. **Smoothing + STE no son decoración:** el paper los ablaciona y muestra que sin ellos el router aprende mal; la decisión dura sola no da gradiente útil. Si implementas tu propio chunker, prueba primero con y sin.
3. **Las etapas externas piden más learning rate.** Ven secuencias más largas y deciden las fronteras de las internas; H-Net escala el LR por etapa (§2.3). Un H-Net con LR uniforme es uno mal sintonizado.
4. **El baseline importa.** H-Net compara contra un Transformer con el **tokenizer GPT-2**; BLT contra **Llama 3 (128K)**. Un tokenizer mejor (o SuperBPE) sube la vara. Al leer «supera a BPE», pregunta *cuál* BPE.
5. **La inicialización por media 'mata' la diversidad.** Todos los tokens nuevos quedan en un punto y el softmax los trata igual; es seguro (no altera la distribución anterior) pero lento. Mide la varianza de los nuevos respecto a los viejos (como en el panel PCA).
6. **Mide en bpb *y* en robustez:** tras un cambio de tokenizer, una init puede ganar en bpb medio y perder en tokens raros o en código. Evalúa en cada dominio.
7. **FOCUS, WECHSEL y compañía fueron diseñados para codificadores/modelos pequeños;** ZeTT documenta que en decodificadores grandes los heurísticos pueden caer «a nivel del azar» en cero-shot y que hacen falta ~cientos de millones de tokens de entrenamiento para cerrar el hueco.
8. **Pregunta incómoda:** ¿tu mejora viene del *tokenizer nuevo* o de *haber entrenado más en el idioma*? El control «seguir con A» responde.

## 11. ⚠️ Errores comunes

* **Comparar pérdida por token** entre tokenizers (mide bpb).
* **Olvidar la matriz de salida** cuando no hay *weight tying*, o los tokens especiales.
* **Inicializar con la media y concluir que 'el método funciona'** sin entrenar: todos los nuevos son iguales hasta que el gradiente los separa.
* **Entrenar el tokenizer nuevo con datos de evaluación** (fuga) o evaluar solo en el idioma destino sin vigilar la regresión en el origen.
* **Creer que H-Net 'demuestra' que BPE ya no sirve:** los resultados son ≤1.3B FLOPs-equivalentes, un conjunto de datos, un baseline BPE concreto.
* **Aplicar el chunker durante el entrenamiento mirando bytes futuros:** rompe la causalidad. Valida con una prueba de perturbación (como la de MiniHNet).

## 12. Autoevaluación

<details>
<summary>1 · (Cálculo) Con $N=4$, calcula $\mathcal{L}_{\text{ratio}}$ para $F=G=0.25$ y para $F=G=0.5$.</summary>

$\frac{4}{3}\big(3\cdot 0.25\cdot0.25 + 0.75\cdot0.75\big)=\frac43(0.1875+0.5625)=1.0$ (el mínimo). Con $F=G=0.5$: $\frac43(3\cdot0.25+0.25)=\frac43\cdot 1.0=1.333$. La pérdida penaliza comprimir menos de lo pedido.

</details>

<details>
<summary>2 · (Cálculo) Un chunk tiene $\hat z_1=(2,0)$ con $P_1=1$ y el siguiente $\hat z_2=(0,4)$ con $P_2=0.5$. ¿Cuánto vale $\bar z_2$?</summary>

$\bar z_1 = P_1\hat z_1 + 0 = (2,0)$. $\bar z_2 = 0.5\cdot(0,4)+0.5\cdot(2,0) = (1, 2)$: una frontera dudosa se mezcla con el chunk anterior.

</details>

<details>
<summary>3 · (Diagnóstico) Tras cambiar de tokenizer, la pérdida por token pasa de 3.1 a 3.4 nats y el equipo dice que el modelo empeoró. Los bytes por token pasaron de 2.5 a 4.0. ¿Qué concluyes?</summary>

bpb antes = 3.1/(ln2·2.5) = 1.79; después = 3.4/(ln2·4.0) = 1.23. El modelo **mejoró** (~31 % menos bits por byte); la pérdida por token no es comparable.

</details>

<details>
<summary>4 · (Diagnóstico) Inicializas con la media y, tras 10 pasos, el modelo reparte la misma probabilidad a todos los tokens nuevos. ¿Es un bug? ¿Cómo lo acelerarías?</summary>

No es un bug: todos los nuevos tienen el mismo embedding y, con weight tying, el mismo logit. Se separan solo cuando el gradiente los diferencia. Acelerar: inicializaciones informadas (FVT/FOCUS/WECHSEL) o añadir ruido de la covarianza de los viejos (Hewitt).

</details>

<details>
<summary>5 · (Transferencia) Políglota quiere añadir 2 000 tokens de dominio (nombres de librerías) al vocabulario de un encoder para embeddings. ¿Qué método elegirías y por qué?</summary>

FVT (media de la descomposición con el tokenizer original) es barato y no necesita datos adicionales: cada librería nueva se inicializa como la media de sus subtokens; luego ajuste corto con el contrastivo del módulo 11. Si tienes corpus del dominio, FOCUS-lite aporta pesos informados. Siempre midiendo retrieval (nDCG@10) y no solo pérdida.

</details>

<details>
<summary>6 · (Transferencia) Escribe en dos líneas qué resultado haría que dejes de decir «la tokenización no está muerta».</summary>

Por ejemplo: una replicación independiente de H-Net/BLT a ≥ 30B parámetros que iguale a un Transformer con un tokenizer fuerte (o que lo supere) **y** con latencia de serving igual o menor. Fíjate que la condición incluye *ingeniería*, no solo bpb.

</details>

<details>
<summary>7 · (Recuerdo) ¿Qué problema resuelven el smoothing y la ratio loss, respectivamente?</summary>

El smoothing hace diferenciable la reconstrucción a partir de decisiones discretas (el gradiente llega al router). La ratio loss evita el colapso a «todo/nada» y fija la compresión objetivo $N$.

</details>

## 13. Referencias

Verificadas (título/autores/fecha) en arXiv o en el repositorio oficial en la fecha de redacción; venues y repositorios revisados de nuevo el 26-oct-2026.

* Hwang, Wang, Gu. **Dynamic Chunking for End-to-End Hierarchical Sequence Modeling** (H-Net). arXiv:2507.07955 (jul. 2025); ICLR 2026. Código: <https://github.com/goombalab/hnet>.
* Pagnoni et al. **Byte Latent Transformer: Patches Scale Better Than Tokens.** arXiv:2412.09871; ACL 2025.
* Videau et al. **From Bytes to Ideas: Language Modeling with Autoregressive U-Nets.** arXiv:2506.14761; NeurIPS 2025.
* Minixhofer et al. **Bolmo: Byteifying the Next Generation of Language Models.** arXiv:2512.15586.
* Neitemeier et al. **SOMBRERO: Measuring and Steering Boundary Placement in End-to-End Hierarchical Sequence Models.** arXiv:2601.22805.
* Minixhofer, Ponti, Vulić. **Zero-Shot Tokenizer Transfer.** NeurIPS 2024, arXiv:2405.07883. Código: <https://github.com/bminixhofer/zett>.
* Dobler, de Melo. **FOCUS: Effective Embedding Initialization for Monolingual Specialization of Multilingual Models.** EMNLP 2023, arXiv:2305.14481.
* Minixhofer, Paischer, Rekabsaz. **WECHSEL: Effective initialization of subword embeddings for cross-lingual transfer of monolingual language models.** NAACL 2022, arXiv:2112.06598.
* Remy, Delobelle, Avetisyan, Khabibullina, de Lhoneux, Demeester. **Trans-Tokenization and Cross-lingual Vocabulary Transfers: Language Adaptation of LLMs for Low-Resource NLP.** COLM 2024, arXiv:2408.04303.
* Gee, Zugarini, Rigutini et al. **Fast Vocabulary Transfer for Language Model Compression.** EMNLP 2022 (industry track; subido a arXiv en 2024), arXiv:2402.09977.
* Liu, Lin, Wang et al. **OFA: A Framework of Initializing Unseen Subword Embeddings for Efficient Large-scale Multilingual Continued Pretraining.** Findings of NAACL 2024, arXiv:2311.08849.
* Minixhofer, Vulić, Ponti. **Universal Cross-Tokenizer Distillation via Approximate Likelihood Matching.** NeurIPS 2025, arXiv:2503.20083.
* Liu, Hayase, Hofmann et al. **SuperBPE: Space Travel for Language Models.** COLM 2025, arXiv:2503.13423.
* Schmidt et al. **Tokenization Is More Than Compression.** EMNLP 2024, arXiv:2402.18376.
* Martins, Astudillo. **From Softmax to Sparsemax: A Sparse Model of Attention and Multi-Label Classification.** ICML 2016.
* Hewitt. **Initializing New Word Embeddings for Pretrained Language Models** (blog, 2021): <https://nlp.stanford.edu/~johnhew/vocab-expansion.html> (espejo: <https://www.cs.columbia.edu/~johnhew/vocab-expansion.html>) (enlace citado en la documentación de `transformers.resize_token_embeddings`).

## 14. Siguiente paso

El **proyecto** (`07_proyecto_hnet_tokenizer_transfer.ipynb`) te pide cambiar el tokenizer de un modelo pequeño, implementar inicializaciones y medir bpb antes/después de un ajuste corto. El **módulo 08** sale del texto: tokens de **imagen y audio**, con otra forma de «tokenizar» (cuantización vectorial).